In [ ]:
# DEVELOPMENT NOTEBOOK — READ BEFORE RUNNING

# This notebook contains all development code and tests for the ML pipeline.
# 
# IMPORTANT:
#   - All modules are imported from the parent folder (flask_app_santiago/)
#   - Any model training in this notebook saves to jupyter_test_arena/test_models/
#   - Production models in flask_app_santiago/models/ are not touched here

: 

In [ ]:
import sys
import os
import numpy as np
import pandas as pd

# Access all modules from the parent folder
sys.path.append("..")

# Override MODELS_DIR for all training in this notebook
# Models are saved to test_models/ not to the production models/ folder
MODELS_DIR = "test_models"

print("sys.path updated — modules accessible from parent folder")
print(f" MODELS_DIR set to: {MODELS_DIR}")

: 

In [ ]:
#Loading Data
DATA_PATH = "../data/merged_final_transformed.csv"
df = pd.read_csv(DATA_PATH)


In [ ]:
# 1 splitter.py


import numpy as np
import pandas as pd

# Global variables and non features. Each model requires only one health var
health_vars = ['BPHIGH', 'CASTHMA', 'COPD',
               'MHLTH', 'PHLTH', 'SLEEP', 'STROKE']
drop_non_features = ["County name", "CountyFIPS",
                     "STATION", "STATION_NAME", "StateAbbr"]


def prepare_data(df: pd.DataFrame, target: str):
    """
    Prepares features X, target y, and year labels from the full dataframe.

    Args:
        df (pd.DataFrame): Full dataset
        target (str):      Target health variable e.g. 'COPD'

    Returns:
        X (pd.DataFrame):     Feature matrix (without target, year, identifiers)
        y (np.ndarray):       Target values
        years (np.ndarray):   Year per row
    """
    df = df.copy()
    drop_health = [col for col in health_vars if col != target]
    df = df.drop(columns=drop_health + drop_non_features)

    # Drop rows where target is NaN. This prevents us from imputing the y variable.
    # In the case of SLEEP this would shrink the dataset only to rows with actual sleep data.
    df = df.dropna(subset=[target])

    X = df.drop(columns=[target, "year"])
    y = df[target].values
    years = df["year"].values  # We need these values to create the folds
    return X, y, years


class Splitter:
    # This class was initially called "Model" but for clarity I changed the name to Splitter.
    # It returns the folds needed for cross validation as well as the full X and y.
    # Cross validation was initially part of this class but was moved to Assessment
    # to prevent confusion and keep responsibilities separate.

    def __init__(self, X: np.ndarray, y: np.ndarray, years: np.ndarray):
        """
        Args:
            X (np.ndarray):     Feature matrix
            y (np.ndarray):     Target values
            years (np.ndarray): Year label per row. Used to create time-based splits.
        """
        self.X = X
        self.y = y
        self.years = years

        # [2013, 2014, ..., 2023]
        self.unique_years = np.sort(np.unique(years))
        self.train_years = self.unique_years[:-1]      # 2013-2022
        self.test_year = self.unique_years[-1]        # 2023

    def time_series_splits(self):
        """
        Each fold adds one more year to training and validates on the next unseen year.
        We start in 2014 because there is very little data for 2013.

        e.g. with years [2014..2022]:
            fold 0: train=2014,           validate=2015
            fold 1: train=2014-2015,      validate=2016
            ...
            fold 7: train=2014-2021,      validate=2022

        The 2023 data corresponds to the test set.

        Returns:
            splits (list of tuples): (X_train, X_val, y_train, y_val) per fold.
            Each row is one fold, and in each fold we have our 4 corresponding datasets.
        """
        splits = []
        for i in range(1, len(self.train_years)):  # Iterating from 2015 to 2022
            train_mask = np.isin(self.years, self.train_years[:i])
            val_mask = self.years == self.train_years[i]

            X_train, y_train = self.X[train_mask], self.y[train_mask]
            X_val,   y_val = self.X[val_mask],   self.y[val_mask]

            splits.append((X_train, X_val, y_train, y_val))

        return splits

    def get_test_split(self):
        """
        Returns full 2013-2022 training data and the held-out 2023 test set.

        Returns:
            X_train, X_test, y_train, y_test (np.ndarray)
        """
        train_mask = np.isin(self.years, self.train_years)
        test_mask = self.years == self.test_year

        return (
            self.X[train_mask], self.X[test_mask],
            self.y[train_mask], self.y[test_mask]
        )


In [ ]:
# 1 test : splitter.py

# Here we check that the general traning and test data is split in the right way 
# and that the folds have also the right structure

# ── Test 1.1: prepare_data ─────────────────────────────────────────────

target = "COPD"
X, y, years = prepare_data(df, target)

print("---- prepare_data ----")
print(f"X shape:      {X.shape}")
print(f"y shape:      {y.shape}")
print(f"years shape:  {years.shape}")
print(f"X columns:    {list(X.columns)}")
print(f"Total rows:   {len(years)}")
print(f"NaNs in y:    {np.isnan(y).sum()}  (should be 0 after dropna)")
assert np.isnan(y).sum() == 0, "FAIL: NaNs found in y"
print("✓ prepare_data correct\n")

X_np     = X.to_numpy()
splitter = Splitter(X_np, y, years)

# ── Test 1.2: time_series_splits ─────────────────────────────────────────────

# Checks that each fold has the right years.
# The size of the training set grows as we add more years.

print("---- time_series_splits ----")
splits = splitter.time_series_splits()
print(f"Number of folds: {len(splits)}")
for i, (X_train, X_val, y_train, y_val) in enumerate(splits):
    train_yrs = np.unique(years[np.isin(years, splitter.train_years[:i+1])])
    val_yr    = splitter.train_years[i+1]
    print(f"  Fold {i}: train={X_train.shape[0]} rows, val={X_val.shape[0]} rows "
          f"| train years={train_yrs}, val year={val_yr}")

assert len(splits) > 0,                    "FAIL: no folds generated"
assert splits[0][0].shape[0] < splits[-1][0].shape[0], \
    "FAIL: training set should grow across folds"
print("time_series_splits correct\n")

# ── Test 1.3: get_test_split ──────────────────────────────────────────────────

# Checks the general training and test structure, not the folds.
# Training should include data from 2013 to 2022.
# Test should only contain data from 2023.

print("---- get_test_split ----")
X_train, X_test, y_train, y_test = splitter.get_test_split()
print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}  |  y_test: {y_test.shape}")

assert 2023 not in np.unique(years[np.isin(years, splitter.train_years)]), \
    "FAIL: 2023 leaked into training data"
assert X_train.shape[0] + X_test.shape[0] == X_np.shape[0], \
    "FAIL: train + test rows don't add up to total rows"
print("2023 isolated as test year")
print("train + test rows add up correctly\n")


---- prepare_data ----
X shape:      (6322, 28)
y shape:      (6322,)
years shape:  (6322,)
X columns:    ['LATITUDE', 'LONGITUDE', 'ELEVATION', 'CLDD', 'DT32', 'DX32', 'DX70', 'DX90', 'EMNT', 'EMXT', 'HTDD', 'PRCP', 'TAVG', 'TMAX', 'TMIN', 'total_population', 'pct_female', 'pct_male', 'median_age', 'pct_bachelors_plus', 'pct_graduate_degree', 'pct_less_than_hs', 'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic', 'median_household_income', 'climate_type_short']
Total rows:   6322
NaNs in y:    0  (should be 0 after dropna)
✓ prepare_data correct

---- time_series_splits ----
Number of folds: 8
  Fold 0: train=255 rows, val=255 rows | train years=[2014], val year=2015
  Fold 1: train=510 rows, val=255 rows | train years=[2014 2015], val year=2016
  Fold 2: train=765 rows, val=255 rows | train years=[2014 2015 2016], val year=2017
  Fold 3: train=1020 rows, val=896 rows | train years=[2014 2015 2016 2017], val year=2018
  Fold 4: train=1916 rows, val=890 rows | train years=[2014 2015

In [ ]:
# 2 preprocessing.py


# This script handles all data preparation before the data is passed to the model.
# It is composed of one standalone function and four classes organized as follows:

# detect_categorical_columns()   Standalone function. Detects which columns are
#                                categorical and returns their indices. This runs
#                                on the DataFrame before converting to numpy.

# SimpleImputer                  Replaces NaN values with column means.

# StandardScaler                 Standardizes features to mean=0 and std=1.

# OneHotEncoder                  Converts categorical columns into binary columns.
#                                Drops first category to avoid multicollinearity.

# Preprocessor                   Coordinates the three classes above through
#                                composition (not inheritance). This means
#                                Preprocessor does not extend the other classes,
#                                it owns instances of them as internal attributes
#                                and delegates work to them in the correct order:
#                                encode, impute,  scale.
#                                This is the only class that should be called
#                                directly from outside this file.

# Preprocessor uses fit_transform() on training data and transform()
# on validation and test data. This separation ensures that no statistics from
# the test set leak into the training process.

import numpy as np
import pandas as pd


# detect_categorical_columns ():

# This function returns the indices of categorical columns in the feature DataFrame.
# In our dataset this will typically only return the index of 'climate_type_short'.
# All other categorical variables (including state) were previously dropped by prepare_data().

def detect_categorical_columns(X_df: pd.DataFrame) -> list:
    """
    Detects categorical column indices from a DataFrame.

    Args:
        X_df (pd.DataFrame): Feature DataFrame before converting to numpy

    Returns:
        cat_col_indices (list): List of integer indices of categorical columns
    """
    cat_cols = X_df.select_dtypes(
        include=["object", "str", "category"]).columns
    return [X_df.columns.get_loc(c) for c in cat_cols]

# SimpleImputer :

# The SimpleImputer replaces NaN values with the column mean.
# Except for SLEEP (52% NaNs), NaNs are below 10% in all other variables.
# It has 3 methods:
#   fit()           — learns the column means from training data only
#   transform()     — replaces NaNs using the learned means (used on val/test)
#   fit_transform() — learns and replaces in one call (used on train)


class SimpleImputer:
    """
    Replaces NaNs with column means computed from training data only.
    Uses np.where + np.take for efficiency instead of looping over columns.
    """

    def fit(self, X: np.ndarray):
        self.means = np.nanmean(X, axis=0)
        # fallback for all-NaN columns — not our case
        self.means[np.isnan(self.means)] = 0

    def transform(self, X: np.ndarray) -> np.ndarray:
        X = X.copy()
        inds = np.where(np.isnan(X))
        X[inds] = np.take(self.means, inds[1])
        return X

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        self.fit(X)
        return self.transform(X)

# StandardScaler:

# The StandardScaler follows the same fit/transform structure as SimpleImputer.
# For each feature it subtracts the mean and divides by the standard deviation,
# producing a distribution with mean=0 and std=1.
# This is important for KRR because the kernel is sensitive to feature scale.


class StandardScaler:
    """
    Standardizes features: z = (x - mean) / std
    """

    def fit(self, X: np.ndarray):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        # avoid division by zero for constant columns
        self.std[self.std == 0] = 1

    def transform(self, X: np.ndarray) -> np.ndarray:
        return (X - self.mean) / self.std

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        self.fit(X)
        return self.transform(X)

# The OneHotEncoder:

# The OneHotEncoder uses the indices returned by detect_categorical_columns().
# fit()      — learns the unique categories from training data
# transform() — builds a numerical array from non-categorical columns,
#               then creates a binary block for each category and merges both
# fit_transform() — does both in one call

# We drop the first category per column to avoid multicollinearity.
# Following the fit/transform separation prevents data leakage. The test set is encoded
# using only the categories seen during training, not its own.


class OneHotEncoder:
    """
    Encodes categorical columns as one-hot arrays.
    Drops first category per column to avoid multicollinearity.
    """

    def fit(self, X: np.ndarray, cat_col_indices: list):
        self.cat_col_indices = cat_col_indices
        self.categories = {}
        for idx in cat_col_indices:
            self.categories[idx] = np.unique(X[:, idx][~pd.isnull(X[:, idx])])

    def transform(self, X: np.ndarray) -> np.ndarray:
        num_cols = [i for i in range(
            X.shape[1]) if i not in self.cat_col_indices]
        X_num = X[:, num_cols].astype(float)

        blocks = []
        for idx in self.cat_col_indices:
            cats = self.categories[idx][1:]  # drop first category
            block = np.stack([(X[:, idx] == c).astype(float)
                             for c in cats], axis=1)
            blocks.append(block)

        return np.hstack([X_num] + blocks) if blocks else X_num

    def fit_transform(self, X: np.ndarray, cat_col_indices: list) -> np.ndarray:
        self.fit(X, cat_col_indices)
        return self.transform(X)

# Preprocessor:

# The Preprocessor coordinates the three components above using composition 
# it owns instances of OneHotEncoder, SimpleImputer and StandardScaler
# and calls them in the correct order: encode, impute, scale.

# fit_transform() is used on the training set — it learns and applies all transformations.
# transform()     is used on val/test sets — it applies using statistics from training only.

# A "fresh" Preprocessor is created inside each CV fold to prevent statistics
# from one fold leaking into another.


class Preprocessor:
    """
    Coordinates encoding, imputation, scaling.
    Fit on training data, transform on val/test.
    """

    def __init__(self, cat_col_indices: list = None):
        self.cat_col_indices = cat_col_indices or []
        self.encoder = OneHotEncoder()
        self.imputer = SimpleImputer()
        self.scaler = StandardScaler()

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        if self.cat_col_indices:
            X = self.encoder.fit_transform(X, self.cat_col_indices)
        X = self.imputer.fit_transform(X)
        X = self.scaler.fit_transform(X)
        return X

    def transform(self, X: np.ndarray) -> np.ndarray:
        if self.cat_col_indices:
            X = self.encoder.transform(X)
        X = self.imputer.transform(X)
        X = self.scaler.transform(X)
        return X


In [ ]:
# ── Test 2.1: detect_categorical_columns ─────────────────────────────────────────────

# Here we test that the categorical colums are indeed detected
#In our case it should only be climate_type_short

target = "STROKE"
X, y, years = prepare_data(df, target)


# Run detection
cat_col_indices = detect_categorical_columns(X)
cat_col_names   = [X.columns[i] for i in cat_col_indices]

print("---- DETECTED CATEGORICAL COLUMNS ----")
print(f"Indices: {cat_col_indices}")
print(f"Names:   {cat_col_names}")
print()


# Print unique values for each detected categorical col
print()
print("---- UNIQUE VALUES PER DETECTED CATEGORICAL COLUMN ----")
for name in cat_col_names:
    uniques = X[name].unique()
    print(f"  {name} ({len(uniques)} unique): {uniques[:10]}")  # cap at 10


---- DETECTED CATEGORICAL COLUMNS ----
Indices: [27]
Names:   ['climate_type_short']


---- UNIQUE VALUES PER DETECTED CATEGORICAL COLUMN ----
  climate_type_short (19 unique): <StringArray>
['ET', 'Cfa', 'BWh', 'BSk', 'Csb', 'Csa', 'Dfc', 'Dfa', 'Am', 'Aw']
Length: 10, dtype: str


In [ ]:
# ── Test 2.2: SimpleImputer ───────────────────────────────────────────────────

# Here we test that after applying the SimpleImputer no NaNs remain in X.
# We first need to encode the categorical variable before imputing,
# since the imputer only works on numerical arrays.

target = "MHLTH"
X, y, years = prepare_data(df, target)

cat_col_indices = detect_categorical_columns(X)
X_np = X.to_numpy()

encoder = OneHotEncoder()
X_np    = encoder.fit_transform(X_np, cat_col_indices).astype(float)

# The shape of X changes after encoding because one-hot adds new columns
print(f"Shape after encoding: {X_np.shape}")
print(f"NaNs before imputing: {np.isnan(X_np).sum()}")

imputer   = SimpleImputer()
X_imputed = imputer.fit_transform(X_np)

# After imputing, NaNs should be 0
print(f"NaNs after imputing:  {np.isnan(X_imputed).sum()}")
assert not np.isnan(X_imputed).any(), "FAIL: NaNs remain after imputation"
print("✓ SimpleImputer works on real data")

Shape after encoding: (6322, 45)
NaNs before imputing: 5999
NaNs after imputing:  0
✓ SimpleImputer works on real data


In [ ]:
# ── Test 2.3: StandardScaler ──────────────────────────────────────────────────

# Here we test that after applying the StandardScaler all features have
# mean=0 and std=1. We use SLEEP as the target since it has the most NaNs.

target = "SLEEP"
X, y, years = prepare_data(df, target)

# Encode and impute first because the scaler requires a clean numerical array
cat_col_indices = detect_categorical_columns(X)
X_np = encoder.fit_transform(X.to_numpy(), cat_col_indices).astype(float)
X_np = SimpleImputer().fit_transform(X_np)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

print(f"Shape: {X_scaled.shape}")
print(f"Means: {X_scaled.mean(axis=0).round(4)}")  # should all be ~0
print(f"Stds:  {X_scaled.std(axis=0).round(4)}")   # should all be ~1

# After scaling, all column means should be 0 and all stds should be 1
assert np.allclose(X_scaled.mean(axis=0), 0, atol=1e-6), "FAIL: means not 0"
assert np.allclose(X_scaled.std(axis=0),  1, atol=1e-6), "FAIL: stds not 1"
print("StandardScaler works on real data")

Shape: (3195, 45)
Means: [-0.  0.  0.  0.  0. -0. -0. -0. -0.  0.  0.  0. -0. -0.  0.  0. -0.  0.
  0.  0. -0.  0. -0. -0.  0. -0. -0.  0.  0. -0.  0. -0.  0. -0.  0. -0.
  0. -0.  0.  0. -0. -0.  0.  0.  0.]
Stds:  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
StandardScaler works on real data


In [ ]:
# ── Test 2.4: OneHotEncoder ───────────────────────────────────────────────────

# Here we test that the OneHotEncoder correctly converts categorical columns
# into binary columns. In our dataset only 'climate_type_short' is encoded.
# We verify two things:
#   1. The output has the correct number of columns after dropping the first category
#   2. The output is fully numerical.

target = "STROKE"
X, y, years = prepare_data(df, target)

cat_col_indices = detect_categorical_columns(X)
X_np = X.to_numpy()

encoder   = OneHotEncoder()
X_encoded = encoder.fit_transform(X_np, cat_col_indices)

print(f"Shape before encoding: {X_np.shape}")
print(f"Shape after encoding:  {X_encoded.shape}  (should have more columns)")
print(f"Detected cat cols:     {[X.columns[i] for i in cat_col_indices]}")
print(f"Categories learned:    { {X.columns[i]: encoder.categories[i] for i in cat_col_indices} }")

# The expected number of columns is:
# original columns categorical columns + (categories per column - 1)
# We subtract 1 because we drop the first category to avoid multicollinearity

expected_new_cols = sum(len(encoder.categories[i]) - 1 for i in cat_col_indices)
expected_shape    = X_np.shape[1] - len(cat_col_indices) + expected_new_cols

assert X_encoded.shape[1] == expected_shape, \
    f"FAIL: expected {expected_shape} cols, got {X_encoded.shape[1]}"
print(f"Column count correct ({expected_shape} expected, first category dropped per col)")

# Checking that all variables ar nummeric after encoding
assert np.issubdtype(X_encoded.dtype, np.floating), \
    "FAIL: non-float values remain after encoding"
print("Output is fully numeric")

Shape before encoding: (6322, 28)
Shape after encoding:  (6322, 45)  (should have more columns)
Detected cat cols:     ['climate_type_short']
Categories learned:    {'climate_type_short': array(['Af', 'Am', 'Aw', 'BSh', 'BSk', 'BWh', 'BWk', 'Cfa', 'Cfb', 'Csa',
       'Csb', 'Dfa', 'Dfb', 'Dfc', 'Dsb', 'Dsc', 'Dwa', 'Dwb', 'ET'],
      dtype=object)}
Column count correct (45 expected, first category dropped per col)
Output is fully numeric


In [ ]:
# ── Test 2.5: Preprocessor ───────────────────────────────────────────────────

# Here we test the full preprocessing pipeline end to end. 
#We inittialy have to split the data before testing the Prerocessor
# The Preprocessor coordinates encoding, imputation and scaling in one call.
# We verify that:
#   1. No NaNs remain after preprocessing
#   2. Training data is correctly standardized (mean=0, std=1)
#   3. Test data uses training statistics — so mean and std will NOT be exactly 0/1

target = "MHLTH"
X, y, years = prepare_data(df, target)

# Split into train and test before preprocessing
cat_col_indices = detect_categorical_columns(X)
X_np     = X.to_numpy()
splitter = Splitter(X_np, y, years)
X_train, X_test, y_train, y_test = splitter.get_test_split()

#Here is where the Preprocessor is actually tested:

# fit_transform() on training data — learns and applies all transformations
# transform() on test data — applies using statistics learned from training only
preprocessor   = Preprocessor(cat_col_indices=cat_col_indices)
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

# Train and test should have same number of columns, different number of rows
print(f"X_train shape: {X_train_scaled.shape}")
print(f"X_test shape:  {X_test_scaled.shape}")

# No NaNs should remain in either dataset 
assert not np.isnan(X_train_scaled).any(), "FAIL: NaNs in X_train"
assert not np.isnan(X_test_scaled).any(),  "FAIL: NaNs in X_test"
print("No NaNs")

# Training data should be fully standardized
assert np.allclose(X_train_scaled.mean(axis=0), 0, atol=1e-6), "FAIL: train mean not 0"
assert np.allclose(X_train_scaled.std(axis=0),  1, atol=1e-6), "FAIL: train std not 1"
print("Train mean=0, std=1")

# Test data will NOT be exactly standardized — this is expected and correct.
# It is scaled using training statistics, not its own,
# so the mean and std will differ slightly from 0 and 1.
print(f"Test mean range: [{X_test_scaled.mean(axis=0).min():.4f}, {X_test_scaled.mean(axis=0).max():.4f}]")
print(f"Test std range:  [{X_test_scaled.std(axis=0).min():.4f}, {X_test_scaled.std(axis=0).max():.4f}]")


X_train shape: (5463, 45)
X_test shape:  (859, 45)
No NaNs
Train mean=0, std=1
Test mean range: [-1.0951, 0.7170]
Test std range:  [0.0507, 1.1549]


In [ ]:
# 3 assessment.py

# This script is in charge of evaluating model performance.
# It contains one class with two methods:

# r2_score()           Measures how much variance the model explains.
#                      1 = perfect, 0 = predicting the mean, < 0 = worse than the mean.

# mean_squared_error() Measures the average squared difference between
#                      predicted and actual values. Used for hyperparameter tuning.

import numpy as np


class Assessment:

    def r2_score(self, y_test: np.ndarray, y_pred: np.ndarray) -> float:
        """
        Calculate the R² score.
        (1) Calculate the mean of true values
        (2) Calculate the total sum of squares (TSS)
        (3) Calculate the residual sum of squares (RSS)
        (4) 1 - (RSS / TSS)

        Args:
            y_test (np.ndarray): True values
            y_pred (np.ndarray): Predicted values

        Returns:
            score (float): R² score — a number <= 1, with 1 being perfect
        """
        mean_y = np.mean(y_test)
        TSS = np.sum((y_test - mean_y) ** 2)
        RSS = np.sum((y_test - y_pred) ** 2)
        return 1 - (RSS / TSS)

    def mean_squared_error(self, y_test: np.ndarray, y_pred: np.ndarray) -> float:
        """
        Calculate the Mean Squared Error (MSE).
        (1) Find the difference between true and predicted values
        (2) Square each difference to eliminate negatives and penalize large errors
        (3) Compute the mean of the squared differences

        Args:
            y_test (np.ndarray): True values
            y_pred (np.ndarray): Predicted values

        Returns:
            score (float): Average squared difference between actual and predicted values
        """
        return np.square(np.subtract(y_test, y_pred)).mean()


In [ ]:
# 4 krr.py
#
# This script implements Kernel Ridge Regression (KRR) from scratch using NumPy.
# It contains one standalone function and one class:
#
# gaussian_kernel()        Standalone function. Computes the RBF kernel matrix
#                          between two sets of points. Measures similarity between
#                          data points

# KernelRidgeRegression    The actual ML model. Uses the kernel matrix to solve
#                          a regularized linear system and make predictions.
#                          It has two methods:
#                            fit()     — learns the dual coefficients from training data
#                            predict() — uses those coefficients to predict new values
#
# The two hyperparameters are:
#   sigma2 — controls the kernel bandwidth. Small sigma2 means only very similar
#             points influence each other. Large sigma2 means more points contribute.
#   lamb   — regularization term. Prevents overfitting by penalizing large coefficients.
#             Too small → overfitting. Too large → underfitting.

import numpy as np

# The Gaussian (RBF) kernel measures similarity between two sets of points.
# For two identical points the kernel value is 1.
# As points move further apart the value decays towards 0.
#
# IMPORTANT : memory optimization vs original implementation:
# Sanjeev's original version computed pairwise distances as:
#
#   diff = x[:, None, :] - y[None, :, :]   # creates (n, m, d) intermediate array
#   sq_dists = np.sum(diff**2, axis=-1)
#
# This is correct but for our dataset of 5754 rows and 45 features it creates
# an intermediate array of shape (5754, 5754, 45) requiring ~12 GB of memory,
# which caused NaNs and crashes during fitting.
#
# We instead use the algebraic identity:
#   ||x - y||^2 = ||x||^2 + ||y||^2 - 2 * x @ y.T

# This only requires an (n, m) matrix — around 265 MB — and produces
# identical results. np.maximum clips any small negative values caused
# by floating point precision before computing the exponential.


def gaussian_kernel(x: np.ndarray, y: np.ndarray, sigma2: float = 2.0) -> np.ndarray:
    """
    Computes the Gaussian (RBF) kernel matrix between arrays x and y.
    Returns an (n, m) matrix where entry (i, j) is:
    exp(-||x_i - y_j||^2 / (2 * sigma2))

    Args:
        x      (np.ndarray): First set of points — shape (n, d)
        y      (np.ndarray): Second set of points — shape (m, d)
        sigma2 (float):      Kernel bandwidth parameter

    Returns:
        K (np.ndarray): Kernel matrix of shape (n, m)
    """
    x = np.atleast_2d(x)
    y = np.atleast_2d(y)

    x_sq = np.sum(x**2, axis=1, keepdims=True)  # (n, 1)
    y_sq = np.sum(y**2, axis=1, keepdims=True)  # (m, 1)
    sq_dists = x_sq + y_sq.T - 2 * (x @ y.T)        # (n, m)
    # clip floating point negatives to 0
    sq_dists = np.maximum(sq_dists, 0)

    return np.exp(-sq_dists / (2 * sigma2))


# KRR works by finding a set of weights (one per training point) called dual coefficients.
# These weights are learned by solving the linear system: (K + λI)a = y
# Where K is the kernel matrix, λ is the regularization term, I is the identity matrix,
# a is the vector of dual coefficients and y is the vector of target values.
# To make predictions on new data we compute the kernel between the new points
# and all training points, then multiply by the dual coefficients.
class KernelRidgeRegression:
    """
    Implements Kernel Ridge Regression using NumPy.
    Uses the Gaussian (RBF) kernel to capture non-linear relationships
    between features and target values.
    """

    def __init__(self, lamb: float = 1e-5, sigma2: float = 0.5):
        """
        Args:
            lamb   (float): Regularization term — prevents overfitting
            sigma2 (float): RBF kernel bandwidth parameter

        These are default values only. The actual values are determined by
        the grid search in main.py and passed in when the model is instantiated.
        """
        self.lamb = lamb
        self.sigma2 = sigma2
        self.coef_ = None  # dual coefficients — learned during fit()
        self.X_train_ = None  # training data — stored for use in predict()

    def fit(self, X_train: np.ndarray, y_train: np.ndarray):
        """
        Learns the dual coefficients by solving (K(X,X) + λI)a = y.

        (1) Compute the symmetric Gram matrix K(X_train, X_train)
        (2) Add regularization: K + λI
        (3) Solve the linear system for the dual coefficients a
        (4) Store X_train — needed to compute the kernel in predict()

        Note: we use np.linalg.solve instead of directly inverting the matrix
        because it is more numerically stable and faster than computing inv(K + λI) @ y.

        Args:
            X_train (np.ndarray): Training feature data
            y_train (np.ndarray): Training target values
        """
        K_train = gaussian_kernel(X_train, X_train, sigma2=self.sigma2)
        self.coef_ = np.linalg.solve(
            K_train + self.lamb * np.eye(len(X_train)), y_train)
        self.X_train_ = X_train

    def predict(self, X_test: np.ndarray) -> np.ndarray:
        """
        Predicts target values for new data using the learned dual coefficients.

        (1) Compute the kernel between test points and all training points
        (2) Multiply by the dual coefficients to get predictions

        Args:
            X_test (np.ndarray): Feature data for making predictions

        Returns:
            y_pred (np.ndarray): Predicted target values
        """
        K_test = gaussian_kernel(X_test, self.X_train_, sigma2=self.sigma2)
        return K_test @ self.coef_


In [ ]:
# ── Test 4 krr.py ─────────────────────────────────────────────────────────────
# Here we test the gaussian_kernel function and the KernelRidgeRegression class.
# We test the following:
#   1. gaussian_kernel — shape, diagonal values and symmetry
#   2. fit()           — dual coefficients are learned and stored correctly
#   3. predict()       — predictions have the right shape and no NaNs
#   4. Full pipeline   — KRR works end to end on real data

target = "STROKE"
X, y, years = prepare_data(df, target)

cat_col_indices = detect_categorical_columns(X)
X_np     = X.to_numpy()
splitter = Splitter(X_np, y, years)

X_train, X_test, y_train, y_test = splitter.get_test_split()

preprocessor   = Preprocessor(cat_col_indices=cat_col_indices)
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

# ── Test 4.1: gaussian_kernel ─────────────────────────────────────────────────
# The kernel matrix should be:
#   - Square and symmetric — K(x, y) == K(y, x)
#   - Diagonal values equal to 1 — a point is identical to itself
#   - All values between 0 and 1 — kernel is a similarity measure
print("-- Test 4.1: gaussian_kernel --")

K = gaussian_kernel(X_train_scaled, X_train_scaled, sigma2=100.0)

print(f"  K shape:          {K.shape}  (should be square: n x n)")
print(f"  K diagonal mean:  {np.diag(K).mean():.6f}  (should be 1.0)")
print(f"  K min:            {K.min():.6f}  (should be >= 0)")
print(f"  K max:            {K.max():.6f}  (should be <= 1)")
print(f"  K symmetric:      {np.allclose(K, K.T)}")

assert K.shape[0] == K.shape[1],          "FAIL: kernel matrix is not square"
assert np.allclose(np.diag(K), 1.0),      "FAIL: diagonal values should be 1"
assert np.allclose(K, K.T),               "FAIL: kernel matrix is not symmetric"
assert K.min() >= 0,                      "FAIL: kernel values should be >= 0"
assert K.max() <= 1 + 1e-10,              "FAIL: kernel values should be <= 1"
print("gaussian_kernel correct\n")

# ── Test 4.2: fit() ───────────────────────────────────────────────────────────
# After fitting:
#   - coef_ should be set and have one value per training point
#   - X_train_ should be stored for use in predict()
#   - No NaNs in the coefficients
print("-- Test 4.2: fit() --")

krr = KernelRidgeRegression(lamb=0.1, sigma2=200.0)
krr.fit(X_train_scaled, y_train)

print(f"  coef_ shape:    {krr.coef_.shape}  (should match training rows)")
print(f"  NaNs in coef_:  {np.isnan(krr.coef_).sum()}  (should be 0)")
print(f"  X_train_ stored: {krr.X_train_ is not None}")

assert krr.coef_ is not None,                        "FAIL: coef_ not set after fit"
assert krr.coef_.shape[0] == X_train_scaled.shape[0],"FAIL: coef_ length should match training rows"
assert not np.isnan(krr.coef_).any(),                "FAIL: NaNs in coef_"
assert krr.X_train_ is not None,                     "FAIL: X_train_ not stored"
print("fit() correct\n")

# ── Test 4.3: predict() ───────────────────────────────────────────────────────
# Predictions should:
#   - Have the same number of rows as X_test
#   - Contain no NaNs
#   - Be in a reasonable range relative to y_test
print("-- Test 4.3: predict() --")

y_pred = krr.predict(X_test_scaled)

print(f"  y_pred shape: {y_pred.shape}  (should match test rows)")
print(f"  NaNs in pred: {np.isnan(y_pred).sum()}  (should be 0)")
print(f"  y_test range: [{y_test.min():.4f}, {y_test.max():.4f}]")
print(f"  y_pred range: [{y_pred.min():.4f}, {y_pred.max():.4f}]")

assert y_pred.shape == y_test.shape, "FAIL: prediction shape mismatch"
assert not np.isnan(y_pred).any(),   "FAIL: NaNs in predictions"
print("predict() correct\n")

# ── Test 4.4: Full pipeline on real data ─────────────────────────────────────
# End to end check — fit and predict on real data and verify scores are sensible
# We use STROKE because it is our best performing variable
print("-- Test 4.4: Full pipeline --")

assessment = Assessment()
mse = assessment.mean_squared_error(y_test, y_pred)
r2  = assessment.r2_score(y_test, y_pred)

print(f"  Test MSE: {mse:.4f}")
print(f"  Test R²:  {r2:.4f}")
print(f"  (STROKE should have R² around 0.65)")

assert mse > 0,  "FAIL: MSE should be positive"
assert r2  > 0,  "FAIL: R² should be positive for STROKE"
print("Full pipeline works on real data\n")

print("---- ALL krr.py TESTS PASSED ----")

-- Test 4.1: gaussian_kernel --
  K shape:          (5463, 5463)  (should be square: n x n)
  K diagonal mean:  1.000000  (should be 1.0)
  K min:            0.001249  (should be >= 0)
  K max:            1.000000  (should be <= 1)
  K symmetric:      True
gaussian_kernel correct

-- Test 4.2: fit() --
  coef_ shape:    (5463,)  (should match training rows)
  NaNs in coef_:  0  (should be 0)
  X_train_ stored: True
fit() correct

-- Test 4.3: predict() --
  y_pred shape: (859,)  (should match test rows)
  NaNs in pred: 0  (should be 0)
  y_test range: [2.0000, 6.4500]
  y_pred range: [2.1239, 4.7165]
predict() correct

-- Test 4.4: Full pipeline --
  Test MSE: 0.1235
  Test R²:  0.6577
  (STROKE should have R² around 0.65)
Full pipeline works on real data

---- ALL krr.py TESTS PASSED ----


In [ ]:
# 5 cross_validator.py

# This script is in charge of running cross validation across time series folds.
# It coordinates the Splitter, Preprocessor, KRR model and Assessment together.

# A fresh Preprocessor is created inside each fold

# start_fold gives us the option to skip early unstable folds where the training set is too small.
# In our data the jump from 255 to 896 rows per year at 2018 makes
# early folds unreliable for hyperparameter selection.

# cross_val_score() returns a dictionary with keys 'mse' and 'r2':

#   scores["mse"] — used for hyperparameter tuning in the grid search
#   scores["r2"]  — used for reporting and interpretation

import numpy as np
from preprocessing import Preprocessor
from assessment import Assessment


class CrossValidator:

    def __init__(self, start_fold: int = 0):
        """
        Args:
            start_fold (int): Only score folds >= this index.
                              By default set to 0 (all folds scored).
        """
        self.start_fold = start_fold
        self.assessment = Assessment()

    def cross_val_score(self, model, splitter, preprocessor: Preprocessor) -> dict:
        """
        Evaluates the model across time series folds using MSE and R².

        Args:
            model:        KRR model — must have fit() and predict() methods
            splitter:     Splitter instance — provides time_series_splits()
            preprocessor: Preprocessor instance — provides cat_col_indices

        Returns:
            dict with keys 'mse' and 'r2' — each a list of scores per scored fold
        """
        # I'll explain the folowing loop because it can be tricky at first
        # The time_series_splits() function from the Splitter class returns a list of tuples:

        # (X_train_fold_0, X_val_fold_0, y_train_fold_0, y_val_fold_0),
        # (...)
        # (X_train_fold_7, X_val_fold_7, y_train_fold_7, y_val_fold_7)

        # Each iteration unpacks one tuple into 4 arrays corresponding to one fold.
        # For each fold we fit_transform on training data and transform only on validation.
        # We are, in other words, iterating across each fold

        mse_scores = []
        r2_scores = []

        for i, (X_train_f, X_val_f, y_train_f, y_val_f) in enumerate(splitter.time_series_splits()):

            # A fresh Preprocessor is created per fold to prevent statistics
            # from one fold leaking into the next
            preprocessor_fold = Preprocessor(
                cat_col_indices=preprocessor.cat_col_indices)
            X_tr = preprocessor_fold.fit_transform(X_train_f)
            X_v = preprocessor_fold.transform(X_val_f)

            model.fit(X_tr, y_train_f)
            y_pred = model.predict(X_v)

            if i >= self.start_fold:
                mse_scores.append(
                    self.assessment.mean_squared_error(y_val_f, y_pred))
                r2_scores.append(self.assessment.r2_score(y_val_f, y_pred))

        return {"mse": mse_scores, "r2": r2_scores}
    
    

In [ ]:
# ── Test 5.1: cross_validator.py  ───────────────────────────────────────────────────

# Here we test that the CrossValidator works correctly by running a grid search
# for the best hyperparameters for a given target variable.
# The best parameters are selected based on the lowest CV MSE across stable folds.

# Recall that the folds have the following structure.
# When we run the cross validation starting in fold 0 we almost always get negative R² values and higher MSE.
# Starting on fold 5 means ultimately doing validations only on 2020, 2021 and 2022 which makes more sense.

#  Fold 0: train=255 rows,  val=255 rows | train years=[2014],                                    val year=2015
#  Fold 1: train=510 rows,  val=255 rows | train years=[2014 2015],                               val year=2016
#  Fold 2: train=765 rows,  val=255 rows | train years=[2014 2015 2016],                          val year=2017
#  Fold 3: train=1020 rows, val=896 rows | train years=[2014 2015 2016 2017],                     val year=2018
#  Fold 4: train=1916 rows, val=890 rows | train years=[2014 2015 2016 2017 2018],                val year=2019
#  Fold 5: train=2806 rows, val=897 rows | train years=[2014 2015 2016 2017 2018 2019],           val year=2020
#  Fold 6: train=3703 rows, val=868 rows | train years=[2014 2015 2016 2017 2018 2019 2020],      val year=2021
#  Fold 7: train=4571 rows, val=892 rows | train years=[2014 2015 2016 2017 2018 2019 2020 2021], val year=2022

# We run two grid searches with STROKE, one starting from fold 0 and one from fold 5
# to compare how start_fold affects hyperparameter selection and scores.


target = "STROKE"
X, y, years = prepare_data(df, target)

cat_col_indices = detect_categorical_columns(X)
X_np     = X.to_numpy()
splitter = Splitter(X_np, y, years)

lamb_grid   = [1e-1, 1.0, 10.0]
sigma2_grid = [100.0, 150.0, 200.0, 300.0]


def run_grid_search(splitter, cat_col_indices, start_fold, label):
    print(f"\n{'='*65}")
    print(f" {label} (start_fold={start_fold})")
    print(f"{'='*65}")

    best_mse    = np.inf
    best_params = {}
    cv          = CrossValidator(start_fold=start_fold)

    for lamb in lamb_grid:
        for sigma2 in sigma2_grid:
            preprocessor = Preprocessor(cat_col_indices=cat_col_indices)
            krr          = KernelRidgeRegression(lamb=lamb, sigma2=sigma2)
            scores       = cv.cross_val_score(krr, splitter, preprocessor)
            mean_mse     = np.mean(scores["mse"])
            mean_r2      = np.mean(scores["r2"])
            print(f"lamb={lamb:.0e}, sigma2={sigma2:6.1f} → CV MSE: {mean_mse:.4f} | CV R²: {mean_r2:.4f}")

            if mean_mse < best_mse and not np.isnan(mean_mse):
                best_mse    = mean_mse
                best_params = {"lamb": lamb, "sigma2": sigma2}

    # Re-run with best params to get final R²
    best_scores = cv.cross_val_score(
        KernelRidgeRegression(**best_params),
        splitter,
        Preprocessor(cat_col_indices=cat_col_indices)
    )
    best_r2 = np.mean(best_scores["r2"])

    print(f"\n Best params: {best_params}")
    print(f" CV MSE: {best_mse:.4f} | CV R²: {best_r2:.4f}")

    return best_params, best_mse, best_r2


# ── Run both and compare ──────────────────────────────────────────────────────

params_fold0, mse_fold0, r2_fold0 = run_grid_search(
    splitter, cat_col_indices, start_fold=0, label="All folds (fold 0 onwards)")

params_fold5, mse_fold5, r2_fold5 = run_grid_search(
    splitter, cat_col_indices, start_fold=5, label="Stable folds only (fold 5 onwards)")

# ── Comparison summary ────────────────────────────────────────────────────────

print(f"\n{'='*65}")
print(f" COMPARISON SUMMARY: STROKE")
print(f"{'='*65}")
print(f"{'':30} {'Fold 0+':>15} {'Fold 5+':>15}")
print(f"{'-'*65}")
print(f"{'Best lamb':<30} {params_fold0['lamb']:>15} {params_fold5['lamb']:>15}")
print(f"{'Best sigma2':<30} {params_fold0['sigma2']:>15} {params_fold5['sigma2']:>15}")
print(f"{'CV MSE':<30} {mse_fold0:>15.4f} {mse_fold5:>15.4f}")
print(f"{'CV R²':<30} {r2_fold0:>15.4f} {r2_fold5:>15.4f}")
print(f"{'='*65}")




 All folds (fold 0 onwards) (start_fold=0)
lamb=1e-01, sigma2= 100.0 → CV MSE: 2.4911 | CV R²: -2.2318
lamb=1e-01, sigma2= 150.0 → CV MSE: 1.9977 | CV R²: -1.0430
lamb=1e-01, sigma2= 200.0 → CV MSE: 1.7966 | CV R²: -0.5554
lamb=1e-01, sigma2= 300.0 → CV MSE: 1.7487 | CV R²: -0.4314
lamb=1e+00, sigma2= 100.0 → CV MSE: 2.1398 | CV R²: -1.3411
lamb=1e+00, sigma2= 150.0 → CV MSE: 1.8829 | CV R²: -0.7256
lamb=1e+00, sigma2= 200.0 → CV MSE: 1.8091 | CV R²: -0.5465
lamb=1e+00, sigma2= 300.0 → CV MSE: 1.7986 | CV R²: -0.5154
lamb=1e+01, sigma2= 100.0 → CV MSE: 2.4396 | CV R²: -1.9615
lamb=1e+01, sigma2= 150.0 → CV MSE: 2.1484 | CV R²: -1.2791
lamb=1e+01, sigma2= 200.0 → CV MSE: 2.0248 | CV R²: -0.9874
lamb=1e+01, sigma2= 300.0 → CV MSE: 1.9297 | CV R²: -0.7606

 Best params: {'lamb': 0.1, 'sigma2': 300.0}
 CV MSE: 1.7487 | CV R²: -0.4314

 Stable folds only (fold 5 onwards) (start_fold=5)
lamb=1e-01, sigma2= 100.0 → CV MSE: 0.1643 | CV R²: 0.5972
lamb=1e-01, sigma2= 150.0 → CV MSE: 0.1384 | C

In [ ]:
# COMPLETE TEST:  FULL PIPELINE: TUNE + EVALUATE EACH TARGET INDEPENDENTLY + FINAL TESTING ON 2023 ─────────────────

# For each health variable we:
# (1) Run a grid search using CrossValidator on stable folds to find best hyperparameters
# (2) Fit the final model on full 2013-2022 training data
# (3) Evaluate on the held-out 2023 test set
# (4) Report MSE and R² for each target

# Note: SLEEP is included here for completeness but its results should not be
# interpreted in the same way as the other variables.
# SLEEP is measured every two  other years which means several folds
# have no data at all. This leaves only 3 usable folds after dropping NaN rows,
# which is not enough for reliable cross validation or hyperparameter selection in this way.

targets     = ['BPHIGH', 'CASTHMA', 'COPD', 'MHLTH', 'PHLTH', 'STROKE', 'SLEEP']
lamb_grid   = [1e-2, 1e-1, 1.0, 10.0]
sigma2_grid = [50.0, 75.0, 100.0, 150.0, 200.0]

final_results = {}

for target in targets:
    print(f"\n{'='*60}")
    print(f"TARGET: {target}")
    print(f"{'='*60}")

    # ── Data preparation ─────────────────────────────────────────
    X, y, years = prepare_data(df, target)
    cat_col_indices = detect_categorical_columns(X)
    X_np     = X.to_numpy()
    splitter = Splitter(X_np, y, years)

    # ── Determine start_fold ──────────────────────────────────────
    # Always use the last 3 stable folds for hyperparameter selection
    n_folds    = len(splitter.time_series_splits())
    start_fold = max(0, n_folds - 3)
    print(f"Total folds: {n_folds} | Scoring from fold: {start_fold}")

    # ── Grid search using CrossValidator ─────────────────────────
    best_mse    = np.inf
    best_r2_cv  = None
    best_params = {}
    cv          = CrossValidator(start_fold=start_fold)

    for lamb in lamb_grid:
        for sigma2 in sigma2_grid:
            preprocessor = Preprocessor(cat_col_indices=cat_col_indices)
            krr          = KernelRidgeRegression(lamb=lamb, sigma2=sigma2)
            scores       = cv.cross_val_score(krr, splitter, preprocessor)

            if len(scores["mse"]) == 0:
                continue

            mean_mse = np.mean(scores["mse"])
            if mean_mse < best_mse and not np.isnan(mean_mse):
                best_mse    = mean_mse
                best_r2_cv  = np.mean(scores["r2"])
                best_params = {"lamb": lamb, "sigma2": sigma2}

    print(f"Best params: {best_params} | Best CV MSE: {best_mse:.4f} | Best CV R²: {best_r2_cv:.4f}")

    # ── Final evaluation on 2023 ──────────────────────────────────
    # Fit on full 2013-2022 training data and evaluate on held-out 2023
    X_train, X_test, y_train, y_test = splitter.get_test_split()

    preprocessor   = Preprocessor(cat_col_indices=cat_col_indices)
    X_train_scaled = preprocessor.fit_transform(X_train)
    X_test_scaled  = preprocessor.transform(X_test)

    krr = KernelRidgeRegression(**best_params)
    krr.fit(X_train_scaled, y_train)
    y_pred = krr.predict(X_test_scaled)

    # Assessment is used here only for final scoring — not for CV
    assessment = Assessment()
    mse = assessment.mean_squared_error(y_test, y_pred)
    r2  = assessment.r2_score(y_test, y_pred)

    final_results[target] = {
        "best_params": best_params,
        "CV_MSE":      round(best_mse, 4),
        "CV_R2":       round(best_r2_cv, 4),
        "Test_MSE":    round(mse, 4),
        "Test_R2":     round(r2, 4)
    }

    print(f"Test MSE: {mse:.4f} | Test R²: {r2:.4f}")
    print(f"y_test=[{y_test.min():.2f}, {y_test.max():.2f}] "
          f"y_pred=[{y_pred.min():.2f}, {y_pred.max():.2f}]")

# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
print(f"\n{'='*85}")
print(f"{'Target':<10} {'lamb':>8} {'sigma2':>8} {'CV MSE':>10} {'CV R²':>10} {'Test MSE':>10} {'Test R²':>10}")
print(f"{'-'*85}")
for target, res in final_results.items():
    print(f"{target:<10} "
          f"{res['best_params']['lamb']:>8} "
          f"{res['best_params']['sigma2']:>8} "
          f"{res['CV_MSE']:>10} "
          f"{res['CV_R2']:>10} "
          f"{res['Test_MSE']:>10} "
          f"{res['Test_R2']:>10}")
print(f"{'-'*85}")

print(f"{'='*85}")


TARGET: BPHIGH
Total folds: 7 | Scoring from fold: 4
Best params: {'lamb': 1.0, 'sigma2': 200.0} | Best CV MSE: 11.4223 | Best CV R²: 0.4292
Test MSE: 14.1886 | Test R²: 0.2772
y_test=[21.45, 52.20] y_pred=[19.59, 40.53]

TARGET: CASTHMA
Total folds: 8 | Scoring from fold: 5
Best params: {'lamb': 0.1, 'sigma2': 200.0} | Best CV MSE: 1.1110 | Best CV R²: -0.2638
Test MSE: 0.6087 | Test R²: 0.3379
y_test=[7.80, 14.60] y_pred=[7.68, 12.53]

TARGET: COPD
Total folds: 8 | Scoring from fold: 5
Best params: {'lamb': 0.01, 'sigma2': 200.0} | Best CV MSE: 1.4501 | Best CV R²: 0.4691
Test MSE: 1.1903 | Test R²: 0.4940
y_test=[3.30, 12.65] y_pred=[2.87, 10.27]

TARGET: MHLTH
Total folds: 8 | Scoring from fold: 5
Best params: {'lamb': 0.1, 'sigma2': 200.0} | Best CV MSE: 7.4519 | Best CV R²: -0.8086
Test MSE: 4.7794 | Test R²: -0.5252
y_test=[13.00, 23.50] y_pred=[10.28, 19.08]

TARGET: PHLTH
Total folds: 8 | Scoring from fold: 5
Best params: {'lamb': 0.01, 'sigma2': 200.0} | Best CV MSE: 3.9510 

In [ ]:
# 6 train.py 

#This script creates and exports the models. Since we are working here in the Jypiter_test_arena 
# the models exported here are saved in a folder called test_models inside this test arena.
# This means that running this code will never overwrite the production models
# that the Flask app uses. those live in flask_app_santiago/models/ and are
# only updated by running the individual file "train.py".


# This script trains and evaluates a Kernel Ridge Regression model for each
# health variable independently. It is run once to generate the fitted models
# which are then loaded by the Flask app for predictions.
#
# These are the steps of the script and the corresponding modules that it calls:

#   (1) Prepares the data and creates time series splits       - splitter.py

#   (2) Runs a grid search using CrossValidator to find
#       the best hyperparameters                               - cross_validator.py
#                                                                 preprocessing.py
#                                                                 krr.py

#   (3) Fits the final model on the full 2013-2022
#       training data                                          - krr.py
#                                                              - preprocessing.py

#   (4) Evaluates on the held-out 2023 test set               - assessment.py

#   (5) Saves the fitted model and preprocessor to disk
#       as a .pkl file                                         - pickle (built-in)


# This file defines:
#   save_model()        — saves a fitted model and preprocessor to disk
#   load_model()        — loads a fitted model and preprocessor from disk
#   tune_and_evaluate() — runs the full pipeline for one target variable
#   print_summary()     — prints a formatted results table
#   main()              — entry point, loops over all target variables
#
# Output: one .pkl file per target variable saved to MODELS_DIR
# These files are loaded by the Flask app via load_model()

import numpy as np
import pandas as pd
import pickle
import os

from splitter import Splitter, prepare_data          # data splitting
from preprocessing import Preprocessor, detect_categorical_columns  # data cleaning
from krr import KernelRidgeRegression           # the ML model
from assessment import Assessment                       # MSE and R² scoring
from cross_validator import CrossValidator                   # CV loop

# ── GLOBAL VARIABLES ─────────────────────────────────────────────────────────
DATA_PATH = "../data/merged_final_transformed.csv"
MODELS_DIR = "test_models"
TARGETS = ['BPHIGH', 'CASTHMA', 'COPD', 'MHLTH', 'PHLTH', 'STROKE', 'SLEEP']
LAMB_GRID = [1e-2, 1e-1, 1.0, 10.0]
SIGMA2_GRID = [50.0, 75.0, 100.0, 150.0, 200.0]


# ── DEFINED IN THIS FILE ──────────────────────────────────────────────────────

def save_model(target: str, krr: KernelRidgeRegression, preprocessor: Preprocessor) -> None:
    """
    Saves the fitted KRR model and preprocessor as a .pkl file.
    Both are saved together so the Flask app can load everything it needs
    in one call without having to refit anything.

    Args:
        target (str):                 Health variable name e.g. 'COPD'
        krr (KernelRidgeRegression):  Fitted model — from krr.py
        preprocessor (Preprocessor):  Fitted preprocessor — from preprocessing.py
    """
    os.makedirs(MODELS_DIR, exist_ok=True)
    path = os.path.join(MODELS_DIR, f"{target}.pkl")
    with open(path, "wb") as f:
        pickle.dump({"krr": krr, "preprocessor": preprocessor}, f)
    print(f"  Saved model → {path}")


def load_model(target: str) -> tuple:
    """
    Loads a fitted KRR model and preprocessor from disk.
    Used by the Flask app to load pre-trained models without retraining.

    Args:
        target (str): Health variable name e.g. 'COPD'

    Returns:
        krr (KernelRidgeRegression): Fitted model
        preprocessor (Preprocessor): Fitted preprocessor
    """
    path = os.path.join(MODELS_DIR, f"{target}.pkl")

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No saved model found for {target} at {path}. "
            f"Run train.py first to generate the model files."
        )

    with open(path, "rb") as f:
        bundle = pickle.load(f)
    return bundle["krr"], bundle["preprocessor"]


def tune_and_evaluate(df: pd.DataFrame, target: str, lamb_grid: list, sigma2_grid: list) -> dict:
    """
    Runs the full training pipeline for a single target variable.
    Calls prepare_data() from splitter.py, CrossValidator from cross_validator.py,
    Preprocessor from preprocessing.py, KernelRidgeRegression from krr.py,
    and Assessment from assessment.py.

    Args:
        df (pd.DataFrame):  Full dataset
        target (str):       Target health variable e.g. 'COPD'
        lamb_grid (list):   Regularisation values to search
        sigma2_grid (list): Kernel bandwidth values to search

    Returns:
        dict: best_params, CV_MSE, CV_R2, Test_MSE, Test_R2
    """
    # ── Data preparation — splitter.py ────────────────────────────
    # prepare_data() from splitter.py
    X, y, years = prepare_data(df, target)
    cat_col_indices = detect_categorical_columns(X)  # from preprocessing.py
    # Splitter from splitter.py
    splitter = Splitter(X.to_numpy(), y, years)

    # ── Determine start_fold ──────────────────────────────────────
    # Always use the last 3 stable folds for hyperparameter selection.
    # Early folds have small training sets and produce unreliable scores.
    n_folds = len(splitter.time_series_splits())
    start_fold = max(0, n_folds - 3)
    print(f"\n  Folds: {n_folds} | Scoring from fold: {start_fold}")

    # ── Grid search — cross_validator.py + krr.py + preprocessing.py ─
    # CrossValidator handles the CV loop and preprocessing per fold.
    # For each combination of lamb and sigma2 it fits the model on each
    # training fold and scores it on the validation fold.
    # Assessment is only used at the end for final test scoring.

    best_mse = np.inf
    best_r2_cv = None
    best_params = {}
    # CrossValidator from cross_validator.py
    cv = CrossValidator(start_fold=start_fold)

    for lamb in lamb_grid:
        for sigma2 in sigma2_grid:
            preprocessor = Preprocessor(
                cat_col_indices=cat_col_indices)  # from preprocessing.py
            krr = KernelRidgeRegression(
                lamb=lamb, sigma2=sigma2)  # from krr.py
            scores = cv.cross_val_score(krr, splitter, preprocessor)

            if len(scores["mse"]) == 0:
                continue

            mean_mse = np.mean(scores["mse"])
            if mean_mse < best_mse and not np.isnan(mean_mse):
                best_mse = mean_mse
                best_r2_cv = np.mean(scores["r2"])
                best_params = {"lamb": lamb, "sigma2": sigma2}

    print(
        f"  Best params: {best_params} | CV MSE: {best_mse:.4f} | CV R²: {best_r2_cv:.4f}")

    # ── Final fit — krr.py + preprocessing.py ────────────────────
    # Refit on the full 2013-2022 training set using the best hyperparameters.
    # This is the model that gets saved and used for Flask predictions.
    X_train, X_test, y_train, y_test = splitter.get_test_split()  # from splitter.py

    preprocessor = Preprocessor(
        cat_col_indices=cat_col_indices)  # from preprocessing.py
    X_train_scaled = preprocessor.fit_transform(X_train)
    X_test_scaled = preprocessor.transform(X_test)

    krr = KernelRidgeRegression(**best_params)  # from krr.py
    krr.fit(X_train_scaled, y_train)
    y_pred = krr.predict(X_test_scaled)

    # ── Save — pickle ─────────────────────────────────────────────
    save_model(target, krr, preprocessor)

    # ── Final evaluation — assessment.py ─────────────────────────
    # Assessment is used here only for final scoring — not for CV
    # from assessment.py
    assessment = Assessment()
    mse = assessment.mean_squared_error(y_test, y_pred)
    r2 = assessment.r2_score(y_test, y_pred)

    print(f"  Test MSE: {mse:.4f} | Test R²: {r2:.4f}")
    print(f"  y_test=[{y_test.min():.2f}, {y_test.max():.2f}] "
          f"y_pred=[{y_pred.min():.2f}, {y_pred.max():.2f}]")

    return {
        "best_params": best_params,
        "CV_MSE":      round(best_mse, 4),
        "CV_R2":       round(best_r2_cv, 4),
        "Test_MSE":    round(mse, 4),
        "Test_R2":     round(r2, 4)
    }


def print_summary(results: dict) -> None:
    """
    Prints a formatted summary table of results across all targets.
    """
    print(f"\n{'='*85}")
    print(f"{'Target':<10} {'lamb':>8} {'sigma2':>8} {'CV MSE':>10} {'CV R²':>10} {'Test MSE':>10} {'Test R²':>10}")
    print(f"{'-'*85}")
    for target, res in results.items():
        print(f"{target:<10} "
              f"{res['best_params']['lamb']:>8} "
              f"{res['best_params']['sigma2']:>8} "
              f"{res['CV_MSE']:>10} "
              f"{res['CV_R2']:>10} "
              f"{res['Test_MSE']:>10} "
              f"{res['Test_R2']:>10}")
    print(f"{'-'*85}")
    print(f"{'Mean':<10} {'':>8} {'':>8} "
          f"{np.mean([v['CV_MSE'] for v in results.values()]):>10.4f} "
          f"{np.mean([v['CV_R2'] for v in results.values()]):>10.4f} "
          f"{np.mean([v['Test_MSE'] for v in results.values()]):>10.4f} "
          f"{np.mean([v['Test_R2'] for v in results.values()]):>10.4f}")
    print(f"{'='*85}")


def main():
    # ── Load data ─────────────────────────────────────────────────
    print("Loading data")
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

    # ── Pipeline for each target ──────────────────────────────

    # tune_and_evaluate() is defined in this file and calls all other modules
    results = {}
    for target in TARGETS:
        print(f"\n{'='*60}")
        print(f"TARGET: {target}")
        print(f"{'='*60}")
        results[target] = tune_and_evaluate(df, target, LAMB_GRID, SIGMA2_GRID)

    # ── Print summary ─────────────────────────────────────────────
    print_summary(results)


if __name__ == "__main__":
    main()
df

Loading data
Loaded 6646 rows, 41 columns

TARGET: BPHIGH

  Folds: 7 | Scoring from fold: 4
  Best params: {'lamb': 1.0, 'sigma2': 200.0} | CV MSE: 11.4223 | CV R²: 0.4292
  Saved model → test_models/BPHIGH.pkl
  Test MSE: 14.1886 | Test R²: 0.2772
  y_test=[21.45, 52.20] y_pred=[19.59, 40.53]

TARGET: CASTHMA

  Folds: 8 | Scoring from fold: 5
  Best params: {'lamb': 0.1, 'sigma2': 200.0} | CV MSE: 1.1110 | CV R²: -0.2638
  Saved model → test_models/CASTHMA.pkl
  Test MSE: 0.6087 | Test R²: 0.3379
  y_test=[7.80, 14.60] y_pred=[7.68, 12.53]

TARGET: COPD

  Folds: 8 | Scoring from fold: 5
  Best params: {'lamb': 0.01, 'sigma2': 200.0} | CV MSE: 1.4501 | CV R²: 0.4691
  Saved model → test_models/COPD.pkl
  Test MSE: 1.1903 | Test R²: 0.4940
  y_test=[3.30, 12.65] y_pred=[2.87, 10.27]

TARGET: MHLTH

  Folds: 8 | Scoring from fold: 5
  Best params: {'lamb': 0.1, 'sigma2': 200.0} | CV MSE: 7.4519 | CV R²: -0.8086
  Saved model → test_models/MHLTH.pkl
  Test MSE: 4.7794 | Test R²: -0.525

In [ ]:
# 7 scenarios.py
#
# This script generates synthetic future data for a given county under a
# predefined climate scenario. It is used by the Flask app to produce
# 10-year health outcome predictions when a user selects a county and scenario.
#
# This file defines:
#   SCENARIOS           — dictionary of available climate scenarios
#   generate_scenario() — builds synthetic future rows for a county
#
# How it works:
#   (1) Takes the 2023 row for the selected county as a baseline
#   (2) Creates horizon copies of that row — one per future year
#   (3) Applies the scenario delta to the relevant climate variable each year
#   (4) Drops all columns not used as features — same as prepare_data() in splitter.py
#   (5) Returns the future feature matrix ready for preprocessor.transform()
#
# Important: generate_scenario() returns raw unscaled features.
#            Always call preprocessor.transform() before passing to krr.predict()
#            Never call preprocessor.fit_transform() — the preprocessor must
#            already be fitted on 2013-2022 training data via load_model()
#
# Note on county uniqueness: county names are not unique across states.
#            For example CLINTON COUNTY exists in Michigan, Iowa and New York.
#            state_abbr is required to uniquely identify a county.

import numpy as np
import pandas as pd

# ── AVAILABLE SCENARIOS ───────────────────────────────────────────────────────

# Each scenario defines which climate variable changes and by how much per year.
# All other features are held constant at their 2023 values for the selected county.
# Add new scenarios here — the Flask app will pick them up automatically.
SCENARIOS = {
    "tavg_increase_0.1": {
        "description": "TAVG increases by 0.1°C per year",
        "variable":     "TAVG",
        "delta_per_year": 0.1
    },
    "tavg_increase_0.5": {
        "description": "TAVG increases by 0.5°C per year",
        "variable":     "TAVG",
        "delta_per_year": 0.5
    }
    # add more scenarios here
}


def generate_scenario(df: pd.DataFrame, county_name: str, state_abbr: str,
                      scenario_key: str, horizon: int = 10):
    """
    Builds synthetic future rows for a county under a given climate scenario.

    Takes the 2023 row for the selected county as a baseline and creates
    one row per future year, applying the scenario delta cumulatively.
    All features other than the scenario variable are held constant at
    their 2023 values.

    Args:
        df (pd.DataFrame):  Full dataset — needed to find the 2023 baseline row
        county_name (str):  Selected county — must match exactly as it appears in the dataset
        state_abbr (str):   State abbreviation e.g. 'MI' — needed because county
                            names are not unique across states
        scenario_key (str): Key from the SCENARIOS dict e.g. 'tavg_increase_0.1'
        horizon (int):      Number of future years to generate — default 10

    Returns:
        X_future (pd.DataFrame): Feature matrix with horizon rows ready for
                                 preprocessor.transform() — NOT yet scaled
        future_years (list):     Corresponding years e.g. [2024, 2025, ..., 2033]
    """
    scenario = SCENARIOS[scenario_key]

    # Filter by both county name and state to get a unique row.
    # County names alone are not unique — e.g. CLINTON COUNTY exists in
    # multiple states. StateAbbr ensures we get exactly one baseline row.
    baseline = df[
        (df["County name"] == county_name) &
        (df["StateAbbr"] == state_abbr) &
        (df["year"] == 2023)
    ].copy()

    #Removing assert and using if and raise
    if len(baseline) == 0:
        raise ValueError(f"No data found for {county_name}, {state_abbr} in 2023.")
    if len(baseline) > 1:
        raise ValueError(f"Found {len(baseline)} rows for {county_name}, {state_abbr} in 2023. Expected exactly 1.")

    future_rows = []
    future_years = list(range(2024, 2024 + horizon))

    for i, year in enumerate(future_years):
        row = baseline.copy()
        row["year"] = year

        # Apply the scenario delta cumulatively
        # Year 1 gets +0.1, year 2 gets +0.2, year 3 gets +0.3 etc.
        row[scenario["variable"]] += scenario["delta_per_year"] * (i + 1)

        future_rows.append(row)

    X_future = pd.concat(future_rows, ignore_index=True)

    # Drop identifier columns and health variables — must match prepare_data() in splitter.py
    # What remains are the 28 feature columns that the model was trained on
    drop_cols = ['year', 'StateAbbr', 'County name', 'CountyFIPS',
                 'STATION', 'STATION_NAME',
                 'BPHIGH', 'CASTHMA', 'COPD', 'MHLTH', 'PHLTH', 'SLEEP', 'STROKE']

    X_future = X_future.drop(columns=drop_cols)

    return X_future, future_years


In [ ]:
# ── 7 Test: scenarios.py ─────────────────────────────────────────────────────

# Here we test that generate_scenario() correctly builds synthetic future rows
# for a given county under a given climate scenario.
# We test the following:
#   5.1 Basic output    — correct shape, years and columns
#   5.2 No leaking cols — health vars and identifiers are dropped
#   5.3 TAVG delta      — increases correctly each year
#   5.4 Other features  — all non-scenario features held constant at 2023 values
#   5.5 Full pipeline   — output flows through preprocessor and KRR correctly

county       = "MOBILE COUNTY"
state        = "AL"              # added — needed to uniquely identify the county
target       = "COPD"
scenario_key = "tavg_increase_0.1"

# ── Test 5.1: Basic output ────────────────────────────────────────────────────
print("-- Test 5.1: Basic output --")
X_future, future_years = generate_scenario(df, county, state, scenario_key, horizon=10)

print(f"  X_future shape:  {X_future.shape}        (should be 10 rows)")
print(f"  Future years:    {future_years}")
print(f"  Columns:         {list(X_future.columns)}")

assert len(X_future)     == 10,   "FAIL: wrong number of rows"
assert len(future_years) == 10,   "FAIL: wrong number of years"
assert future_years[0]   == 2024, "FAIL: should start at 2024"
assert future_years[-1]  == 2033, "FAIL: should end at 2033"
print("Shape and years correct\n")

# ── Test 5.2: No leaking columns ──────────────────────────────────────────────
print("-- Test 5.2: No leaking columns --")
forbidden = ['BPHIGH', 'CASTHMA', 'COPD', 'MHLTH', 'PHLTH', 'SLEEP', 'STROKE',
             'year', 'StateAbbr', 'County name', 'CountyFIPS', 'STATION', 'STATION_NAME']
leaked = [c for c in forbidden if c in X_future.columns]
assert not leaked, f"FAIL: these columns should not be in X_future: {leaked}"
print("No forbidden columns in output\n")

# ── Test 5.3: TAVG delta applied correctly ────────────────────────────────────
# TAVG should increase by 0.1 each year from the 2023 baseline
print("-- Test 5.3: TAVG delta --")
baseline_tavg = df[
    (df["County name"] == county) &
    (df["StateAbbr"]   == state)  &
    (df["year"]        == 2023)
]["TAVG"].values[0]

print(f"  Baseline TAVG (2023): {baseline_tavg:.4f}")
for i, row in X_future.iterrows():
    expected = baseline_tavg + 0.1 * (i + 1)
    actual   = row["TAVG"]
    assert np.isclose(actual, expected), \
        f"FAIL: TAVG wrong at year {future_years[i]} — expected {expected:.4f}, got {actual:.4f}"
    print(f"  Year {future_years[i]}: TAVG={actual:.4f}  expected={expected:.4f}")
print("TAVG increases by 0.1 each year\n")

# ── Test 5.4: Non-scenario features unchanged ─────────────────────────────────
# All features except TAVG should be identical to the 2023 baseline row
print("-- Test 5.4: Non-scenario features unchanged --")
baseline_row = df[
    (df["County name"] == county) &
    (df["StateAbbr"]   == state)  &
    (df["year"]        == 2023)
].drop(columns=forbidden, errors="ignore")

for col in X_future.columns:
    if col == "TAVG":
        continue  # skip — this one should change
    baseline_val = baseline_row[col].values[0]
    future_vals  = X_future[col].values
    assert (future_vals == baseline_val).all(), \
        f"FAIL: {col} changed across future rows — should be constant at {baseline_val}"
print("All non-scenario features held constant at 2023 values\n")

# ── Test 5.5: Full pipeline compatibility ─────────────────────────────────────
# X_future should flow through preprocessor.transform() and krr.predict() without errors
print("-- Test 5.5: Full pipeline --")
krr, preprocessor = load_model(target)
X_scaled = preprocessor.transform(X_future.to_numpy())
y_pred   = krr.predict(X_scaled)

print(f"  y_pred shape: {y_pred.shape}  (should be (10,))")
print(f"  Predictions:  {y_pred.round(4)}")
print(f"  Trend:        {'↑ increasing' if y_pred[-1] > y_pred[0] else '↓ decreasing'}")

assert y_pred.shape == (10,),          "FAIL: wrong prediction shape"
assert not np.isnan(y_pred).any(),     "FAIL: NaNs in predictions"
print("Scenario feeds into model correctly\n")

print("=== ALL scenarios.py TESTS PASSED ===")

-- Test 5.1: Basic output --
  X_future shape:  (10, 28)        (should be 10 rows)
  Future years:    [2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033]
  Columns:         ['LATITUDE', 'LONGITUDE', 'ELEVATION', 'CLDD', 'DT32', 'DX32', 'DX70', 'DX90', 'EMNT', 'EMXT', 'HTDD', 'PRCP', 'TAVG', 'TMAX', 'TMIN', 'total_population', 'pct_female', 'pct_male', 'median_age', 'pct_bachelors_plus', 'pct_graduate_degree', 'pct_less_than_hs', 'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic', 'median_household_income', 'climate_type_short']
Shape and years correct

-- Test 5.2: No leaking columns --
No forbidden columns in output

-- Test 5.3: TAVG delta --
  Baseline TAVG (2023): 21.6000
  Year 2024: TAVG=21.7000  expected=21.7000
  Year 2025: TAVG=21.8000  expected=21.8000
  Year 2026: TAVG=21.9000  expected=21.9000
  Year 2027: TAVG=22.0000  expected=22.0000
  Year 2028: TAVG=22.1000  expected=22.1000
  Year 2029: TAVG=22.2000  expected=22.2000
  Year 2030: TAVG=22.3000  expected=22